# Building a World-Aware Agent with Claude

Claude has a knowledge cutoff. Ask it "what's the probability of a US recession?" and it will either hedge or give you stale data from its training set.

This cookbook shows how to give Claude **real-time world awareness** by injecting calibrated prediction market data into the system prompt. The agent knows today's geopolitical risk level, oil prices, recession probability, and Fed rate expectations — not from web search (which returns narratives), but from **probability data backed by real money**.

We use [SimpleFunctions](https://simplefunctions.dev/world), which aggregates 9,706 prediction market contracts into an ~800-token world state snapshot. No API key needed.

**What you'll build:**
1. System prompt injection with live world state
2. Tool use for focused topic drilling and market search
3. MCP server integration for Claude Desktop / Claude Code

## Setup

In [1]:
%pip install anthropic requests -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import requests
import anthropic

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var

## Step 1: Fetch the World State

One API call. ~800 tokens of markdown. Covers geopolitics, economy, energy, elections, crypto, tech. Anchor contracts (recession, Fed, Iran invasion) always appear. Updated every 15 minutes from 9,706 prediction markets on Kalshi (CFTC-regulated) and Polymarket.

In [3]:
world = requests.get("https://simplefunctions.dev/api/agent/world").text
print(world)

# World — Apr 10, 2026
Regime: Energy deflation — oil crashing, watch recession signals

SF Index v2: Disagree 39/100 | GeoRisk 26/100 | Breadth +0.40 | Activity 100/100

Markets: SPY $675.11 (+2.33%) | QQQ $605.15 (+0.08%) | IWM $260.23 (+2.63%) | VIXY $30.55 (-9.72%) | SHY $82.40 (+0.04%) | IEF $95.37 (+0.05%) | TLT $86.81 (-0.08%) | HYG $80.19 (+0.59%) | GLD $434.54 (-1.69%) | USO $124.56 (-9.62%) | UNG $11.10 (-3.81%) | CPER $35.24 (+3.37%) | UUP $27.54 (-0.72%) | FXE $107.68 (+0.62%) | XLE $58.05 (-3.51%) | EEM $60.42 (+5.68%) | IBIT $40.46 (+2.98%) | ETHE $17.97 (+4.6%)

## Edges
- [32h] Which countries will Donald Trump visit in 2026?: China: 93c, no (24c edge, spread:1c) [polymarket]
- [32h] Iran x Israel/US conflict ends by...?: June 30: 87c, no (23c edge, spread:1c) [polymarket]
- [32h] Iran x Israel/US conflict ends by...?: May 15: 76c, no (23c edge, spread:1c) [polymarket]
- [32h] Iran x Israel/US conflict ends by...?: April 30: 70c, no (22c edge, spread:1c) [polymarket]
- 

Every line is a fact with a number. `Iran invasion: 53c (+5c)` means a 53% probability priced by people with real money at stake. This is calibrated data, not "tensions remain elevated."

## Step 2: World-Aware Claude via System Prompt

Inject the world state into the system prompt. Claude now has current data to cite instead of hallucinating.

In [4]:
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system=f"""You are a macro research analyst with real-time world awareness.

Use the data below as ground truth. Cite specific probabilities and prices.
Do not hallucinate numbers — if it's not in the data, say you don't know.

{world}""",
    messages=[
        {"role": "user", "content": "What's the current geopolitical risk level and how does it affect energy markets?"}
    ],
)

print(response.content[0].text)

# Geopolitical Risk & Energy Markets — Apr 10, 2026

## Current GeoRisk Reading
**GeoRisk Index: 26/100** — Notably low. This is a "risk-off geopolitics" environment, meaning markets are not pricing significant conflict escalation premiums.

---

## Key Geopolitical Signals

### Iran/Israel-US Conflict (Primary Energy Risk Vector)
Prediction markets are pricing **rapid de-escalation**:

| Resolution Date | Probability |
|----------------|-------------|
| April 15 | **65c** |
| April 30 | **70c** |
| May 15 | **76c** |
| June 30 | **87c** |

The market already anchors **US x Iran ceasefire at 100c for April 7** — meaning that event is essentially confirmed resolved. The conflict-ends probabilities above represent clean closure/formalization.

### Russia/Ukraine
Ceasefire by April 30: **only 5c** — war continues but appears largely priced in with minimal fresh shock potential.

---

## Energy Market Impact

| Asset | 24h Move | Signal |
|-------|----------|--------|
| USO (Oil) | **-9.62

## Step 3: Tool Use for Deeper Data

The system prompt gives a panoramic view. For specific questions, give Claude tools to drill deeper.

In [5]:
tools = [
    {
        "name": "get_world_state",
        "description": "Get current world state from prediction markets. Use focus parameter to concentrate the token budget on specific topics for deeper coverage.",
        "input_schema": {
            "type": "object",
            "properties": {
                "focus": {
                    "type": "string",
                    "description": "Comma-separated topics: geopolitics, economy, energy, elections, crypto, tech. Empty for all."
                }
            }
        }
    },
    {
        "name": "get_world_delta",
        "description": "Get only what changed in the world since a given time. ~30-50 tokens vs 800 for full state. Use for periodic refresh.",
        "input_schema": {
            "type": "object",
            "properties": {
                "since": {
                    "type": "string",
                    "description": "Time window: 30m, 1h, 6h, 24h"
                }
            },
            "required": ["since"]
        }
    },
    {
        "name": "search_prediction_markets",
        "description": "Search for specific prediction market contracts. Returns prices, volumes, spreads from Kalshi and Polymarket.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Natural language search, e.g. 'iran oil' or 'fed rate cut'"
                }
            },
            "required": ["query"]
        }
    }
]


def handle_tool_call(name, input_data):
    """Execute a tool call and return the result."""
    if name == "get_world_state":
        url = "https://simplefunctions.dev/api/agent/world"
        if input_data.get("focus"):
            url += f"?focus={input_data['focus']}"
        return requests.get(url).text
    elif name == "get_world_delta":
        return requests.get(
            f"https://simplefunctions.dev/api/agent/world/delta?since={input_data['since']}"
        ).text
    elif name == "search_prediction_markets":
        resp = requests.get(
            f"https://simplefunctions.dev/api/public/scan?q={input_data['query']}&limit=10"
        )
        return json.dumps(resp.json().get("markets", [])[:10], indent=2)
    return "Unknown tool"

In [6]:
# Agentic loop with tool use
messages = [
    {"role": "user", "content": "Deep dive on Iran risk — search for specific contracts and tell me what the market thinks."}
]

system_prompt = f"""You are a macro research analyst with real-time world awareness.
Cite specific numbers from prediction markets. Use your tools to search for contracts.

{world}"""

while True:
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system=system_prompt,
        tools=tools,
        messages=messages,
    )

    # Check if Claude wants to use tools
    if response.stop_reason == "tool_use":
        # Process tool calls
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = handle_tool_call(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })
                print(f"Tool: {block.name}({block.input})")

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
    else:
        # Final response
        for block in response.content:
            if hasattr(block, "text"):
                print("\n" + block.text)
        break

Tool: search_prediction_markets({'query': 'Iran conflict ceasefire'})
Tool: search_prediction_markets({'query': 'Iran nuclear deal'})
Tool: search_prediction_markets({'query': 'Iran regime collapse overthrow'})

Here's the full Iran risk picture from prediction markets — across three dimensions: **conflict resolution**, **nuclear diplomacy**, and **regime stability**.

---

## 🇮🇷 Iran Risk Deep Dive — Apr 10, 2026

### 1. 🔫 Conflict Resolution Timeline
*"Iran x Israel/US conflict ends by...?" — Polymarket | $11.3M+ total volume*

| Deadline | Price | Implied Probability |
|---|---|---|
| April 7 | **57¢** | 57% ✅ *(near-expired, likely resolving)* |
| April 15 | **65¢** | 65% |
| April 30 | **74¢** | 74% |
| May 15 | **76¢** | 76% |
| June 30 | **87¢** | 87% |
| December 31 | **95¢** | 95% |

**What the market is saying:** The conflict is broadly expected to end — it's not a question of *if*, but *when*. The 8-point gap between April 30 (74¢) and June 30 (87¢) implies roughly a **13% p

## Step 4: MCP Integration

SimpleFunctions exposes all tools via MCP. For Claude Desktop or Claude Code, connect with one command:

```bash
claude mcp add simplefunctions --url https://simplefunctions.dev/api/mcp/mcp
```

This gives Claude 40 tools including `get_world_state`, `get_world_delta`, `scan_markets`, and more — no code needed.

For Claude Desktop, add to your MCP config:

```json
{
  "mcpServers": {
    "simplefunctions": {
      "url": "https://simplefunctions.dev/api/mcp/mcp"
    }
  }
}
```

## Why Prediction Markets?

| | Web Search | News API | Prediction Markets |
|---|---|---|---|
| **Output** | Narrative text | Headlines | Calibrated probabilities |
| **Token cost** | 2,000-5,000 | 500-1,000 | ~800 for everything |
| **Latency** | 2-5 seconds | 500ms | ~200ms (cached) |
| **Calibration** | None | None | Participants lose money when wrong |

A price of 53c on "Iran invasion" encodes the aggregate judgment of everyone with money at risk. A headline encodes what an editor thought would get clicks.

**Links:**
- [World State API](https://simplefunctions.dev/api/agent/world) — try it now (no auth)
- [MCP Server](https://simplefunctions.dev/api/mcp/mcp) — 40 tools
- [Documentation](https://simplefunctions.dev/docs/guide)